# Clase 6 - Aprendizaje Supervisado III

## SVM y SGDClassifier con dataset de uso de redes sociales Gen Z

En esta notebook trabajo con el dataset `genz_social_media_usage_1M.csv`. El objetivo es resolver un problema de clasificacion supervisada: predecir el nivel de adiccion a redes sociales (`addiction_level`) a partir de variables de uso digital y bienestar personal.

Modelos utilizados:

- Support Vector Machine lineal (`LinearSVC`)
- `SGDClassifier`

La idea es entrenar ambos modelos, evaluar sus resultados y comparar cual funciona mejor para este dataset.

Importacion de librerias

Primero se importan las librerias necesarias para leer el dataset, preparar las variables, entrenar los modelos y evaluar los resultados.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score

pd.set_option("display.max_columns", None)

Carga y exploracion inicial

Se carga el archivo CSV y se revisan las primeras filas, las dimensiones y la distribucion de la variable objetivo.

In [ ]:
archivo = "genz_social_media_usage_1M.csv"
datos = pd.read_csv(archivo)

print("Dimensiones originales:", datos.shape)
display(datos.head())

print("\nColumnas del dataset:")
print(datos.columns.tolist())

print("\nDistribucion de la variable objetivo:")
print(datos["addiction_level"].value_counts())
print("\nPorcentaje por clase:")
print(round(datos["addiction_level"].value_counts(normalize=True) * 100, 2))

Preparacion del dataset

Como el archivo tiene 1 millon de registros, se usa una muestra de 20.000 filas para que el entrenamiento sea mas rapido. Despues se separan las variables predictoras (`X`) y la variable objetivo (`y`).

In [ ]:
muestra = datos.sample(n=20000, random_state=42)
print("Dimensiones de la muestra usada:", muestra.shape)

X = muestra.drop(columns=["addiction_level"])
y = muestra["addiction_level"]

columnas_numericas = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
columnas_categoricas = X.select_dtypes(include=["object"]).columns.tolist()

print("\nVariables numericas:", columnas_numericas)
print("Variables categoricas:", columnas_categoricas)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTamaño entrenamiento:", X_train.shape)
print("Tamaño prueba:", X_test.shape)

In [ ]:
try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=True)

preprocesamiento = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), columnas_numericas),
        ("cat", encoder, columnas_categoricas)
    ]
)

Se entrenan dos modelos vistos en el bloque de aprendizaje supervisado. En este caso uso SVM lineal y SGDClassifier.

In [ ]:
modelo_svm = Pipeline(steps=[
    ("preprocesamiento", preprocesamiento),
    ("modelo", LinearSVC(C=1.0, max_iter=5000, random_state=42))
])

modelo_sgd = Pipeline(steps=[
    ("preprocesamiento", preprocesamiento),
    ("modelo", SGDClassifier(loss="log_loss", max_iter=2000, tol=1e-3, random_state=42))
])

modelos = {
    "SVM lineal": modelo_svm,
    "SGDClassifier": modelo_sgd
}

Evaluacion

Para comparar los modelos se usa accuracy, F1 macro, matriz de confusion y reporte de clasificacion. Estas metricas ayudan a ver no solo cuantos casos acierta el modelo, sino tambien como trabaja con cada clase.

In [ ]:
resultados = []
predicciones = {}

for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    predicciones[nombre] = y_pred

    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average="macro")
    resultados.append({"Modelo": nombre, "Accuracy": acc, "F1 macro": f1_macro})

    print("\n" + "=" * 60)
    print(nombre)
    print("Accuracy:", round(acc, 4))
    print("F1 macro:", round(f1_macro, 4))
    print("\nReporte de clasificacion:")
    print(classification_report(y_test, y_pred, digits=4))

resumen = pd.DataFrame(resultados).sort_values(by="Accuracy", ascending=False)
print("\nResumen comparativo:")
display(resumen)

Las matrices de confusion permiten ver los aciertos y errores por clase: `Low`, `Medium` y `High`.

In [ ]:
clases = ["Low", "Medium", "High"]

for nombre, y_pred in predicciones.items():
    matriz = confusion_matrix(y_test, y_pred, labels=clases)
    disp = ConfusionMatrixDisplay(confusion_matrix=matriz, display_labels=clases)
    disp.plot(values_format="d")
    plt.title(f"Matriz de confusion - {nombre}")
    plt.show()

Comparacion visual

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(resumen["Modelo"], resumen["Accuracy"])
plt.ylim(0, 1)
plt.title("Comparacion de accuracy")
plt.ylabel("Accuracy")
plt.xlabel("Modelo")
plt.show()

## Resultados obtenidos en la prueba

Con una muestra de 20.000 registros y una division 80/20 entre entrenamiento y prueba, los resultados principales fueron:

| Modelo | Accuracy | F1 macro |
|---|---:|---:|
| SVM lineal | 0.9898 | 0.9877 |
| SGDClassifier | 0.9875 | 0.9854 |

Los dos modelos tuvieron buen rendimiento, pero SVM lineal quedo apenas por encima.

El modelo SVM lineal busca separar las clases mediante un hiperplano, intentando encontrar una frontera de decision clara entre los grupos. El SGDClassifier, en cambio, ajusta sus parametros mediante un proceso iterativo de optimizacion, lo que lo hace muy util cuando se trabaja con datasets grandes.

En los resultados obtenidos, ambos modelos tuvieron un rendimiento alto. Sin embargo, el modelo con mejor accuracy fue el que aparece primero en la tabla comparativa. Para este problema, ese modelo seria el mas conveniente porque logro clasificar mejor los niveles `Low`, `Medium` y `High` dentro de la muestra utilizada.

In [ ]:
mejor_modelo = resumen.iloc[0]
print("El modelo con mejor accuracy fue:", mejor_modelo["Modelo"])
print("Accuracy:", round(mejor_modelo["Accuracy"], 4))
print("F1 macro:", round(mejor_modelo["F1 macro"], 4))